In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
f = open("/content/drive/MyDrive/파일위치/book_law_data.txt", 'r', encoding='cp949')
sList = []
lines = f.readlines()
for line in lines:
    line = line.strip()  # 줄 끝의 줄 바꿈 문자를 제거한다.
    sList.append(line)
f.close()
print(sList[:5])
print(len(sList))

['부정 청탁, 문제 유출, 입학 취소', '임금표, 임금등급, 노동기준을 사업장에 미공표', '가등기의 개념', '북한 학자들이 쓴 저서 또는 논문의 입수가 어려운 현 단계에서 그들의 국제법에 대한 태도를 개관한다는 것은 불가능한 일이다.', '그러나 몇 편의 논문 및 사전류114 등을 보건대 그들의 서술방식이 특이함을 느끼게 된다.']
1949154


In [3]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 15.8 MB/s eta 0:00:00


In [ ]:
from konlpy.tag import Kkma

kkma = Kkma()

In [4]:
def tokenize_sentence(sentence):
        tokenized_sentence = kkma.morphs(sentence)
        l = ''
        for s in tokenized_sentence:
            s = re.sub(r"[^가-힣\s]", " ", s)
            l += s +' '
        return l

In [ ]:
from tqdm import tqdm
import re
tokenized_data =[]
for i in tqdm(range(len(sList))):
    tokenized_data.append(tokenize_sentence(sList[i]))
    if (i!= 0) and (i % 5000==0):
        f = open("/content/drive/MyDrive/파일위치/book_law_data_kkm.txt",'a')
        for s in tokenized_data:
            f.write(s+'\n')
        tokenized_data =[]
        f.close()
print(i)
f = open("/content/drive/MyDrive/파일위치/book_law_data_kkm.txt",'a')
for s in tokenized_data:
    f.write(s+'\n')
f.close()

In [6]:
f = open("/content/drive/MyDrive/파일위치/book_law_data_kkm.txt", 'r')
sList = []
lines = f.readlines()
for line in lines:
    line = line.strip()  # 줄 끝의 줄 바꿈 문자를 제거한다.
    sList.append(line)
f.close()
print(sList[:5])

['부정 청탁   문제 유출   입학 취소', '임금 표   임금 등급   노동 기준 을 사업장 에 미 공표', '가 아 등기 의 개념', '북한 학자 들 이 쓰   저서 또는 논문 의 입수 가 어렵   현 단계 에서 그 들 의 국제법 에 대하   태도 를 개관 하  다는 것 은 불가능 하   일 이 다', '그러나 몇 편의 논문 및 사전 류     등 을 보 건대 그 들 의 서술 방식 이 특이 함 을 느끼 게 되  다']


In [7]:
result = [s.split(' ') for s in sList]
print(result[:3])

[['부정', '청탁', '', '', '문제', '유출', '', '', '입학', '취소'], ['임금', '표', '', '', '임금', '등급', '', '', '노동', '기준', '을', '사업장', '에', '미', '공표'], ['가', '아', '등기', '의', '개념']]


In [8]:
for l in result:
    while True:
        try:
            l.remove('')
        except:
            break
print(result[:3])

[['부정', '청탁', '문제', '유출', '입학', '취소'], ['임금', '표', '임금', '등급', '노동', '기준', '을', '사업장', '에', '미', '공표'], ['가', '아', '등기', '의', '개념']]


In [9]:
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

model = Word2Vec(result, vector_size=100, window=5, min_count=1, workers=4, sg=0)
model.save('/content/drive/MyDrive/모델위치/book_law_kkm_w2v.model')

In [11]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/파일위치/pan_qna_kkm.csv')
df.tail()

,question,answer,embedding
84153,모집 설립 어서 설립 경과 조사 절차 어서 설명 주 시 시오,모집설립에 있어서 설립경과의 조사는 발기설립의 경우와 거의 동일합니다. 모집설립에 ...,[-2.38549814e-01 2.11867094e-02 9.72366035e-...
84154,발기 설립 어서 변태 설립 사항 조사 설명 주 시 시오,발기설립에 있어서 정관으로 상법 제290조 각 호의 변태설립사항을 정한 때에는 이사...,[-2.32459411e-01 -4.19367775e-02 1.01206744e+...
84155,발기 설립 어서 설립 경과 조사 절차 설명 주 시 시오,이사와 감사는 취임후 지체없이 회사의 설립에 관한 모든 사항이 법령 또는 정관의 규...,[-4.16801900e-01 -7.41601065e-02 1.07209158e+...
84156,발기 설립 임원 선임 절차 어떻 게 는가요,발기인이 인수한 주식에 대하여 상법 제295조의 규정에 의한 납입과 현물출자의 이행...,[ 7.15333372e-02 6.66695386e-02 9.00789678e-...
84157,정관 효력 발생 요건 무엇 가요,"정관은 원칙적으로 공증인의 인증을 받음으로써 효력이 생깁니다. 다만, 소규모 회사 ...",[ 1.70083299e-01 -3.04049671e-01 1.10665154e+...


In [14]:
#임베딩하는 코드
##시용자 입력이 들어오면 (들어올 때마다) 사용자 입력과 전체 질문리스트(혹은 해당 카테고리의 질문 리스트)를 임베딩해줌
from tqdm import tqdm
def get_document_vectors(document_list):
    document_embedding_list = []
    #각 문서에 대해서
    for line in tqdm(document_list):
        doc2vec = None
        count = 0
        for word in line.split():
            #print(word)
            try:
                w = model.wv[word]
                count += 1

                #해당 문서에 있는 모든 단어들의 벡터 값을 더한다.
                if doc2vec is None :
                    doc2vec = w
                else :
                    doc2vec = doc2vec + w
            except:
                pass

        if doc2vec is not None:
            # 단어 벡터를 모두 더한 벡터의 값을 문서 길이로 나눠준다.
            doc2vec = doc2vec/count
            document_embedding_list.append(doc2vec)
        else:
            document_embedding_list.append(None)

    # 각 문서에 대한 문서 벡터 리스트를 리턴
    return document_embedding_list


In [15]:
emd = get_document_vectors(df['question'])

100%|██████████| 84158/84158 [00:04<00:00, 18408.98it/s]


In [16]:
df['embedding'] = emd
df = df.dropna()
df.to_csv('/content/drive/MyDrive/파일위치/pan_qna_kkm_w2v.csv', index=False)